# Error Models: Underestimated Uncertainties

In this tutorial, we will build on our [previous demonstration](LVM-getting-started.html) using simulated data to consider a case in which we are given data and uncertainties, but we believe the uncertainties are systematically underestimated for certain pixels. This issue sometimes appears in modeling stellar spectra, when telluric features, sky lines, or other issues are not fully accounted for in the uncertainties. We will demonstrate how to incorporate an array of parameters to handle this by adding an additional variance term to the likelihood, sometimes called a "jitter" or "excess variance" term.

As usual, we will start with some standard imports and set up the simulated data.

In [ ]:
import jax
import matplotlib.pyplot as plt
import numpy as np
import numpyro.distributions as dist

import pollux as plx
from pollux.models.transforms import LinearTransform, ScatterTransform

jax.config.update("jax_enable_x64", True)
%matplotlib inline

## Generating simulated data

We will generate data for 2048 stars, with a latent dimensionality of 8, 2 labels, and 128 pixels in the spectra. We will follow the same prescription as in the [previous tutorial](LVM-getting-started.html) to generate the simulated labels and spectra. After generating the data, we will then add in a systematic error (as a function of pixel number) that is not accounted for in the reported uncertainties.

In [ ]:
from helpers import make_simulated_linear_data

n_stars = 2048  # number of simulated stars to generate in the train and test sets
n_latents = 8  # size of the latent vector per star
n_labels = 2  # number of labels to generate per star
n_flux = 128  # number of spectral flux pixels per star

rng = np.random.default_rng(seed=42)

A = np.zeros((n_labels, n_latents))
A[0, 0] = 1.0
A[1, 1] = 1.0

B = rng.normal(scale=0.1, size=(n_flux, n_latents))
B[:, 0] = B[:, 0] + 4 * np.exp(-0.5 * (np.arange(n_flux) - n_flux / 2) ** 2 / 5**2)
B[:, 1] = B[:, 1] + 2 * np.exp(-0.5 * (np.arange(n_flux) - n_flux / 4) ** 2 / 3**2)

data, truth = make_simulated_linear_data(
    n_stars=n_stars,
    n_latents=n_latents,
    n_flux=n_flux,
    n_labels=n_labels,
    A=A,
    B=B,
    rng=rng,
)

# Now we add a periodic systematic error to the flux:
true_systematic_err = 2.0 * (np.cos(2 * np.pi * np.arange(n_flux) / (n_flux / 4))) ** 2
data["flux"] = rng.normal(data["flux"], scale=true_systematic_err)

The systematic error we add inflates the uncertainties significantly in a periodic pattern with pixel number:

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(true_systematic_err)
plt.ylabel("Systematic error")
plt.xlabel("Spectral pixel")

With simulated data in hand, we now proceed to run the model on this data.

As with the previous tutorial, we will package this data (to prepare for using it in  {py:class}`~pollux.models.LVM`) by defining a {py:class}`~pollux.data.PolluxData` instance with the data. We use the standard shift-and-scale normalization for the spectral flux data and labels (as shown in the previous tutorial):

In [ ]:
all_data = plx.data.PolluxData(
    flux=plx.data.OutputData(
        data["flux"],
        err=data["flux_err"],
        preprocessor=plx.data.ShiftScalePreprocessor.from_data(data["flux"]),
    ),
    label=plx.data.OutputData(
        data["label"],
        err=data["label_err"],
        preprocessor=plx.data.ShiftScalePreprocessor.from_data(data["label"]),
    ),
).preprocess()

For this example, we will again use the model in a "supervised" or "train and apply" mode, in which we will train the model on a subset of the data and then apply it to the remaining data. We will use the first 1024 stars for training and the remaining 1024 stars for testing (since they are not ordered in any way):

In [ ]:
train_data = all_data[: n_stars // 2]
test_data = all_data[n_stars // 2 :]
len(train_data), len(test_data)

## Constructing the model

In our first demonstration, we will use the same model as in the previous tutorial (i.e. without adding any additional parameters to learn the systematic error). We will then show that the model performs worse than a model that accounts for the (unknown) systematic error by simultaneously learning this vector.

### Model 1: no systematic error (same as in the previous tutorial)

In [ ]:
model1 = plx.LVM(latent_size=8)
model1.register_output("label", LinearTransform(output_size=n_labels))
model1.register_output("flux", LinearTransform(output_size=n_flux))

In [ ]:
# opt_pars1, svi_results1 = model1.optimize(
#     train_data,
#     rng_key=jax.random.PRNGKey(112358),
#     optimizer=numpyro.optim.Adam(1e-3),
#     num_steps=32768,
#     svi_run_kwargs={"progress_bar": False},
# )
# svi_results1.losses.block_until_ready()[-1]
results1 = model1.optimize_iterative(
    train_data, rng_key=jax.random.PRNGKey(112358), max_cycles=128, progress=False
)
opt_pars1 = results1.params

We now evaluate the model on the test data and compare the results to the true labels.

In [ ]:
fixed_pars1 = model1.output_pars(opt_pars1)

test_results1 = model1.optimize_iterative(
    test_data,
    rng_key=jax.random.PRNGKey(12345),
    max_cycles=128,
    fixed_pars=fixed_pars1,
    blocks=["latents"],
    progress=False,
)
test_opt_pars1 = test_results1.params

In [ ]:
predict_test_values1 = model1.predict_outputs(
    fixed_pars1, latents=test_opt_pars1["latents"]
)

In [ ]:
pt_style = {"ls": "none", "ms": 2.0, "alpha": 0.5, "marker": "o", "color": "k"}

fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for i in range(predict_test_values1["label"].shape[1]):
    axes[i].plot(
        predict_test_values1["label"][:, i], test_data["label"].data[:, i], **pt_style
    )
    axes[i].set(xlabel=f"Predicted label {i}", ylabel=f"True label {i}")
    axes[i].axline([0, 0], slope=1, color="tab:green", zorder=-100)
_ = fig.suptitle("Test set: predicted vs. true labels", fontsize=22)

It looks like the model is doing a reasonable job of recovering the true labels, but the prediction error (variance) is large for the test set labels. We will now demonstrate how to improve this by adding a parameter to learn the systematic error.

### Model 2: an inferred vector of extra flux uncertainties

We will now add a vector parameter to the model to learn the systematic error at each spectral pixel. We do this with a {py:class}`~pollux.models.transforms.ScatterTransform`, which adds a fitted per-pixel scatter `s` in quadrature to the reported errors: $\sqrt{\sigma^2 + s^2}$. It is passed as the output's `err_transform`, so `s` is fitted alongside everything else. We will set the prior on this parameter to be a half-Normal distribution (a normal truncated at 0) with a mean of 0 and a standard deviation of 5 (i.e. we expect the systematic error to be small but allow the possibility of it being large). 

In [ ]:
err_trans = ScatterTransform(output_size=n_flux, priors={"s": dist.HalfNormal(5.0)})

We now define the model as we did before, but pass in the transform of the uncertainties we defined in the previous cell when defining the "flux" output:

In [ ]:
model2 = plx.LVM(latent_size=8)
model2.register_output(
    "flux", LinearTransform(output_size=n_flux), err_transform=err_trans
)

# We register the label output as before, but we could have also added an unknown
# systematic uncertainty here
model2.register_output("label", LinearTransform(output_size=n_labels))

We now optimize the model. Because both outputs are linear in the latents, we use
{py:meth}`~pollux.models.LVM.optimize_iterative`, which solves those blocks exactly. The
scatter `s` is the exception: it enters the likelihood through the variance rather than
the mean, so there is no closed form for it and that block is fitted with SVI. We give it
a generous number of steps per cycle with `block_options`:

In [ ]:
results2 = model2.optimize_iterative(
    train_data,
    rng_key=jax.random.PRNGKey(112358),
    max_cycles=128,
    progress=False,
    block_options={"flux:err": {"num_steps": 5000}},
)
opt_pars2 = results2.params

And then optimize and evaluate the model on the test data:

In [ ]:
fixed_pars2 = model2.output_pars(opt_pars2)

test_results2 = model2.optimize_iterative(
    test_data,
    rng_key=jax.random.PRNGKey(12345),
    max_cycles=128,
    fixed_pars=fixed_pars2,
    blocks=["latents"],
    progress=False,
)
test_opt_pars2 = test_results2.params

In [ ]:
predict_test_values2 = model2.predict_outputs(
    fixed_pars2, latents=test_opt_pars2["latents"]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for i in range(predict_test_values2["label"].shape[1]):
    axes[i].plot(
        predict_test_values2["label"][:, i], test_data["label"].data[:, i], **pt_style
    )
    axes[i].set(xlabel=f"Predicted label {i}", ylabel=f"True label {i}")
    axes[i].axline([0, 0], slope=1, color="tab:green", zorder=-100)
_ = fig.suptitle("Test set: predicted vs. true labels", fontsize=22)

We can also compare the inferred systematic error parameter to the true systematic error:

In [ ]:
inferred_s = all_data["flux"].preprocessor.inverse_transform_err(
    opt_pars2["flux"]["err"]["s"]
)

plt.figure(figsize=(6, 4))
plt.plot(true_systematic_err, label="True systematic error", lw=2, color="k")
plt.plot(inferred_s, label="Inferred systematic error", color="tab:orange")
plt.xlabel("Spectral pixel")
plt.ylabel("Systematic error")
plt.legend(loc="lower left")

## When the inferred scatter collapses to zero

If you look closely at the figure above, the inferred scatter tracks the truth well at almost every pixel, but a couple of pixels are pinned near zero where they should be near 2. This is actually an important failure mode to understand for any model that fits a variance.

First, a pixel whose residuals are much smaller than its own reported uncertainty is
suspicious: the model shouldn't be able to fit the data better than the noise allows.

In [ ]:
predict_train2 = model2.predict_outputs(opt_pars2)
resid2 = np.asarray(predict_train2["flux"]) - np.asarray(train_data["flux"].data)

resid_scatter = resid2.std(axis=0)
reported = np.asarray(train_data["flux"].err).mean(axis=0)
fitted_s = np.asarray(opt_pars2["flux"]["err"]["s"])

# a pixel fitted far better than its reported uncertainty allows
suspect = np.where(resid_scatter < 0.5 * reported)[0]

print(f"suspicious pixels: {suspect}")
print()
print(f"{'pixel':>6} {'rms(resid)':>12} {'reported err':>14} {'fitted s':>11}")
for pix in [*suspect, 16, 48]:
    flag = "  <-- collapsed" if pix in suspect else "  (other example pixels)"
    print(
        f"{pix:>6} {resid_scatter[pix]:12.5f} {reported[pix]:14.5f} "
        f"{fitted_s[pix]:11.5f}{flag}"
    )

At the other example pixels, the fitted `s` is essentially equal to the scatter of the
residuals, which is exactly what it should be. At the collapsed pixels the model has
fitted the data better than the reported uncertainties permit, and `s` has gone to
zero.

### Why this happens

For each data point, the (Gaussian) log-likelihood contributes

$$
-\frac{1}{2}\left[\frac{r^2}{\sigma^2 + s^2} + \ln\left(\sigma^2 + s^2\right)\right]
$$

and this is unbounded above: as the residual $r \to 0$ and $s \to 0$ together, the
first term stays finite while the second diverges to $+\infty$. So there is an infinitely good "solution" sitting at $s = 0$ for any pixel the model can fit exactly. (This is the same pathology as a component of a Gaussian mixture model collapsing onto a single data point.)

This happens because of the way the `optimize_iterative` method works. If `s` at some pixel is small, that pixel's inverse variance is enormous, so the latents solve step weights it far above all the others. Each star has 8 free latent values and 130 observations, which is plenty of freedom to fit one over-weighted pixel almost exactly. The residual there shrinks, which justifies an even smaller `s`, and so it collapses to zero. 

The pixels that are most prone to this failure are ones that start with a small `s`. `optimize_iterative` starts from a random draw of the parameters from the priors, and our prior on `s` is deliberately wide. So if a pixel starts with a small `s`, it is likely to collapse to zero. In this case, we could avoid the problem by either changing the initialization of `s`, or by using a different optimization method that does not have this failure mode. For example, we could use the `optimize` method, which optimizes all parameters simultaneously with SVI. This method is often slower to converge, but it does not have the same failure mode.

In [ ]:
s_draws = np.asarray(dist.HalfNormal(5.0).sample(jax.random.PRNGKey(0), (n_flux,)))
print(f"draws from HalfNormal(5) span {s_draws.min():.3f} to {s_draws.max():.3f}")
print(f"{(s_draws < 0.1).sum()} of {n_flux} pixels start below 0.1")

### One fix: start with a better initialization

Nothing is wrong with the model, we just need to start from a better initialization. Here, we'll start all of the values of the scatter at 1.0:

In [ ]:
initial_params2 = {
    "latents": opt_pars1["latents"],
    "flux": {
        "data": opt_pars1["flux"]["data"],
        "err": {"s": np.ones(n_flux)},
    },
    "label": {"data": opt_pars1["label"]["data"], "err": {}},
}

results2b = model2.optimize_iterative(
    train_data,
    rng_key=jax.random.PRNGKey(112358),
    max_cycles=128,
    progress=False,
    initial_params=initial_params2,
    block_options={"flux:err": {"num_steps": 5000}},
)
opt_pars2b = results2b.params

In [ ]:
inferred_s_warm = all_data["flux"].preprocessor.inverse_transform_err(
    opt_pars2b["flux"]["err"]["s"]
)

plt.figure(figsize=(7, 4.5))
plt.plot(true_systematic_err, label="True systematic error", lw=2, color="k")
plt.plot(inferred_s_warm, label="Inferred, warm-started", color="tab:orange")
plt.xlabel("Spectral pixel")
plt.ylabel("Systematic error")
_ = plt.legend(loc="lower left")

The collapsed scatter values are gone!

A general lessons to take from this is that whenever a model fits a variance along with a mean, the likelihood has these degenerate optima. An optimizer that solves sub-problems exactly and iterates over sub-problems is more likely to fall into one of these failure modes. 

To summarize, we have demonstrated how to incorporate a systematic error term into the model to account for underestimated uncertainties in the data. We added a parameter to capture this for the spectral fluxes, per pixel. But we could have instead added a single value of the error inflation (i.e. for all pixels), or added a similar parameter for the label data. This can significantly improve the model's ability to accurately predict label values, as demonstrated on simulated data. 

More complex modifications of the models or additional parameters (e.g., adding a simultaneous model of the continuum flux shape) can also be incorporated, but that requires implementing a custom numpyro model. We will demonstrate this in a subsequent tutorial.